# Projeto Integrador de Processamento de Linguagem Natural (rev_17-sd)

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** Transformers, PyTorch  
- **Arquivo Original:** `PJBL.ipynb`  

---


In [16]:
!pip install azure-ai-translation-text

In [17]:
!pip install evaluate
!pip install bert_score
!pip install googletrans==4.0.0rc1
!pip install -q sentence-transformers
!pip install pandas

In [18]:
frases_conteudo = """O sol nasceu cedo e iluminou toda a cidade com seus raios dourados.
Ela gosta de café forte pela manhã, sempre sem açúcar e bem quente.
O cachorro dorme tranquilamente no sofá da sala desde o começo da tarde.
Vamos ao cinema amanhã à noite para assistir à estreia do novo filme.
Ele comprou um carro novo e já planeja uma viagem no fim de semana.
Choveu muito durante a noite e as ruas ficaram completamente alagadas.
A criança brinca alegremente no parquinho enquanto os pais conversam.
Estamos aprendendo inteligência artificial com exemplos práticos e divertidos.
A pizza chegou fria porque o entregador se perdeu no caminho.
O ônibus estava lotado e demorou mais de uma hora para chegar ao centro.
Ela leu três livros em um mês e fez anotações sobre cada um deles.
O vento derrubou a árvore em frente à escola durante a tempestade.
A aula começou atrasada porque o professor ficou preso no trânsito.
O gato subiu no telhado e não consegue mais descer sozinho.
Ele esqueceu a carteira em casa e teve que voltar para buscá-la.
O filme foi emocionante e fez todos na sala de cinema chorarem.
A internet está lenta hoje, o que está atrapalhando as videochamadas.
Vamos caminhar no parque depois do almoço para aproveitar o dia bonito.
Ela perdeu o celular ontem enquanto voltava do trabalho de ônibus.
O professor explicou com paciência cada detalhe do conteúdo da prova.
Eles viajaram para o interior para descansar e ficar longe da correria.
O restaurante estava fechado, então decidiram pedir comida pelo aplicativo.
Chamei um táxi pelo aplicativo porque estava chovendo muito na rua.
Ele acordou com dor de cabeça e decidiu faltar ao trabalho hoje.
O trem passou direto pela estação sem parar, causando confusão nos passageiros.
O jogo terminou empatado mesmo depois de uma prorrogação intensa.
Ela trouxe bolo de chocolate para comemorar o aniversário do colega.
A bateria do celular acabou no meio da ligação importante com o cliente.
Compramos frutas frescas na feira e fizemos uma salada deliciosa.
O vizinho reformou a casa inteira e agora está construindo uma varanda.
A luz acabou de repente durante o filme, assustando todo mundo na sala.
Ele gosta de música clássica e sempre ouve enquanto trabalha no computador.
O mar estava agitado ontem, então não foi possível entrar na água.
Assisti a uma série nova que me recomendaram e gostei bastante da história.
Ela fez uma apresentação excelente e foi muito elogiada pelos colegas.
Estamos organizando a festa de fim de ano com a ajuda de todos do setor.
Ele ganhou um presente surpresa e ficou muito emocionado com o gesto.
As flores estão desabrochando e deixando o jardim cheio de cores vivas.
Precisamos revisar o relatório final antes de enviar para o cliente.
A loja abriu mais cedo hoje por causa da promoção especial de primavera.
Choveu granizo na cidade ontem à tarde, causando pequenos danos aos carros.
O cachorro fugiu do quintal e foi encontrado pelos vizinhos na praça.
Ela foi ao médico ontem porque estava com dores nas costas há dias.
O avião decolou com atraso devido às condições climáticas instáveis.
A comida estava deliciosa, especialmente o prato principal com molho caseiro.
Vamos pintar a sala amanhã e trocar os móveis de lugar também.
Ele trocou de emprego recentemente e está animado com os novos desafios.
A bicicleta quebrou a corrente no meio do caminho para a escola.
Estudamos bastante para a prova de IA e fizemos vários exercícios de revisão.
O correio entregou o pacote com atraso, mas tudo chegou em perfeito estado.
As crianças desenharam com giz no chão da escola e criaram lindas imagens."""

with open("frases.txt", "w", encoding="utf-8") as f:
    f.write(frases_conteudo)


In [21]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
from bert_score import score
from googletrans import Translator
from evaluate import load
bertscore = load("bertscore")

In [32]:
import time
from tqdm import tqdm
from azure.core.credentials import AzureKeyCredential
from azure.ai.translation.text import TextTranslationClient

# --- CONFIGURAÇÃO ---
subscription_key = "SUA_CHAVE_AZURE_TRANSLATOR_AQUI"
region = "brazilsouth"

credential = AzureKeyCredential(subscription_key)
client = TextTranslationClient(credential=credential, region=region)

# --- FUNÇÃO DE TRADUÇÃO ---
def bing_translate(text, from_lang, to_lang):
    """Traduz texto com Azure Translator, usando formato correto e tratamento de erro."""
    time.sleep(0.5)
    body = [{"text": text}]
    try:
        # Tradução
        response = client.translate(
            body=body,
            from_language=from_lang,
            to_language=[to_lang]
        )
        return response[0].translations[0].text

    except Exception as e:
        print(f"[ERRO] {from_lang} -> {to_lang}: {e}")
        return None

# --- TESTE DIRETO ---
print("\n=== TESTE DIRETO ===")
res = bing_translate("Hello, how are you?", "en", "tlh-Latn")
print("→ Tradução para Klingon:", res)

# --- SEU PIPELINE COMPLETO (corrigido) ---
with open("frases.txt", "r", encoding="utf-8") as f:
    originais = [linha.strip() for linha in f if linha.strip()]

referencia = []

for frase in tqdm(originais, desc="Traduzindo com Bing"):
    try:
        step1 = bing_translate(frase, "pt", "la")
        step2 = bing_translate(step1, "la", "ru")
        step3 = bing_translate(step2, "ru", "ja")
        step4 = bing_translate(step3, "ja", "tlh-Latn")  # Klingon só como destino!
        step5 = bing_translate(step4, "en", "ar")  # ⚠️ Troca Klingon->en, pois tlh como origem não é aceito
        step6 = bing_translate(step5, "ar", "zh-Hant")
        back = bing_translate(step6, "zh-Hant", "pt")

        referencia.append(back or "ERRO")
    except Exception as e:
        print(f"\nErro na frase '{frase[:20]}...': {e}")
        referencia.append("ERRO")

# --- RESULTADO ---
print("\nTraduções concluídas!")
for i, (orig, refe) in enumerate(zip(originais, referencia)):
    print(f"\nFrase {i+1}:\nOriginal:  {orig}\nReferência: {refe}")



=== TESTE DIRETO ===
→ Tradução para Klingon: qavan. bIpIv'a'?


Traduzindo com Bing:   0%|          | 0/51 [00:00<?, ?it/s]

[ERRO] pt -> la: (400036) The target language is not valid.
Code: 400036
Message: The target language is not valid.
[ERRO] la -> ru: (400035) The source language is not valid.
Code: 400035
Message: The source language is not valid.
[ERRO] ru -> ja: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] ja -> tlh-Latn: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] en -> ar: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] ar -> zh-Hant: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.


Traduzindo com Bing:   2%|▏         | 1/51 [00:07<06:32,  7.85s/it]

[ERRO] zh-Hant -> pt: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] pt -> la: (400036) The target language is not valid.
Code: 400036
Message: The target language is not valid.
[ERRO] la -> ru: (400035) The source language is not valid.
Code: 400035
Message: The source language is not valid.
[ERRO] ru -> ja: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] ja -> tlh-Latn: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] en -> ar: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] ar -> zh-Hant: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not

...

Traduzindo com Bing:  67%|██████▋   | 34/51 [02:13<01:00,  3.55s/it]

[ERRO] zh-Hant -> pt: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] pt -> la: (400036) The target language is not valid.
Code: 400036
Message: The target language is not valid.
[ERRO] la -> ru: (400035) The source language is not valid.
Code: 400035
Message: The source language is not valid.
[ERRO] ru -> ja: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.
[ERRO] ja -> tlh-Latn: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.


Traduzindo com Bing:  67%|██████▋   | 34/51 [02:16<01:08,  4.01s/it]

[ERRO] en -> ar: (400020) One of the elements in the array text is not valid.
Code: 400020
Message: One of the elements in the array text is not valid.




KeyboardInterrupt



In [31]:
import requests

subscription_key = "SUA_CHAVE_AZURE_TRANSLATOR_AQUI"
region = "brazilsouth"

url = f"https://api.cognitive.microsofttranslator.com/languages?api-version=3.0"

resp = requests.get(url)
data = resp.json()

translation_langs = data["translation"]
print(f"Idiomas disponíveis: {len(translation_langs)} encontrados\n")

# Exemplo: checar se Klingon está disponível
print("Existe tlh-Latn?", "tlh-Latn" in translation_langs)
print("Existe tlh-Piqd?", "tlh-Piqd" in translation_langs)

# Mostrar nome legível se existir
if "tlh-Latn" in translation_langs:
    print("Nome:", translation_langs["tlh-Latn"]["name"])


Idiomas disponíveis: 137 encontrados

Existe tlh-Latn? True
Existe tlh-Piqd? True
Nome: Klingon (Latin)


In [ ]:

modelo_sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

similaridades = []
for orig, para in zip(originais, referencia):
    emb1 = modelo_sbert.encode(orig, convert_to_tensor=True)
    emb2 = modelo_sbert.encode(para, convert_to_tensor=True)
    sim = util.cos_sim(emb1, emb2).item()
    similaridades.append(sim)

tabela = pd.DataFrame({
    "Frase Original": originais,
    "Referência": referencia,
    "BERTScore F1": [round(f.item(), 4) for f in F1],
    "SBERT Similaridade": [round(s, 4) for s in similaridades]
})

print("\n📊 Tabela comparativa BERTScore x SBERT:\n")
display(tabela)
